# Model 2 sequence inputs (JDSearch)

Turns each slate's history into model-ready sequences for Model 2 (DIN), one channel per action type, plus candidate-level inputs.


## Why the history is split into three channels

The history is not one continuous log. Each action type was recorded over a different window before the search (full dataset, `history_days_before`):

| Action | Recorded for | Evidence |
|---|---|---|
| CLICK | last **7 days** | 99% within 6.5 days; 107 of 8.3M clicks after day 7 |
| CART, FLW | last **90 days** | 99% within 88–89 days; 1,204 of 11.5M carts after day 90 |
| ORD | last **365 days** | spread over the year, median age 154 days |

Within the first week the mix looks like a normal funnel (about 6 clicks per cart and 7 carts per order); beyond day 7 clicks are simply missing, and beyond day 90 only orders remain. The cut-offs at exactly 7 / 90 / 365 days also confirm that the time gaps are in seconds.

Consequences for the inputs:

- **One mixed sequence is misleading.** Recent clicks are dense (about 48 per user in one week), so keeping the most recent N actions fills the sequence with a few days of clicks and cuts the older orders, which are the long-term signal this model is for.
- **Action types and age are confounded.** In a mixed sequence, "old" almost always means "order". Cross-type weights (CLICK = 1 … ORD = 10) compare a week of clicks with a year of orders.

So each user's history is stored as **three channels**, each with one consistent observation window: **click** (7 days), **cart** (CART and FLW, 90 days), **order** (365 days). Each channel has its own length cap and its own recency scale. Within a channel, older simply means older.

## Rules shared with Model 1 and the scalar features

All feature pipelines apply the same rules, so the models see the same history:

- **Windows:** CLICK ≤ 7 days, CART and FLW ≤ 90 days, ORD ≤ 365 days before the search (boundary inclusive).
- **All history is kept**, including actions just before the search. Actions minutes before a search were checked for being copies of the search's own interactions: no history event carries the current search's query, and the overlap with candidates decays smoothly with age instead of jumping near zero, so they are treated as genuine recent behaviour, as in Model 1 (Section 1.4.1 reports the dependence as a limitation). `MIN_AGE_DAYS = 0`; setting it above 0 reproduces the filtered variant for a sensitivity check.
- **Missing attribute codes:** a brand or category of -1 in the product file means "unknown", not a real brand, and maps to the unknown index.
- **Train-only statistics:** vocabularies, caps and recency scales come from the training split only; val1, val2 and test are encoded with them unchanged.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: set DATA_DIR to a local folder in the Settings cell.")

Mounted at /content/drive


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

## Settings
Each value below is a modelling decision that should be justified in the report. The per-channel caps and recency half-lives are not set by hand: they are computed from the train split in Step 4.3 and saved in `vocab.json`.

In [ ]:
# Project folders (same layout as every other notebook; Drive is mounted in the cell above)
PROJECT = Path("/content/drive/MyDrive/Grp8Submission/pipeline_data")
DATA = PROJECT / "step1"                    # step 1 outputs
# Stop early if the shared folder is not visible (otherwise new empty folders would be
# created in your own My Drive): add a shortcut to the shared Grp8Submission folder in My Drive.
for d in (PROJECT, DATA):
    assert d.is_dir(), f"{d} not found: add a shortcut to the shared Grp8Submission folder in My Drive"
assert (DATA / "train_sampled.parquet").exists(), f"step 1 outputs not found in {DATA}"
DATA_DIR = DATA                              # step 1 outputs
OUT_DIR = PROJECT / "step2_model2_seq"       # Model 2 sequence inputs go here (read by step 3 and step 4)
OUT_DIR.mkdir(exist_ok=True)

# Step 1 outputs, one file per split (70 / 10 / 10 / 10). Vocabularies, caps and recency
# scales are fitted on "train" only; the other splits are encoded with them unchanged.
SPLIT_FILES = {"train": "train_sampled.parquet",   # negative-sampled training slates
               "val1": "val1.parquet",             # early stopping / tuning (full slates)
               "val2": "val2.parquet",             # calibration and the gate (full slates)
               "test": "test.parquet"}             # final evaluation (full slates)

# History channels: action types and the window (days) over which they were recorded
CHANNELS = {
    "click": (["CLICK"], 7),
    "cart":  (["CART", "FLW"], 90),
    "order": (["ORD"], 365),
}
MIN_AGE_DAYS = 0.0    # keep all history (team decision, same as Model 1); set e.g. 0.01 to drop
                      # actions < ~15 min before the search for a sensitivity check
CAP_PERCENTILE = 75   # per channel: keep the most recent actions up to the 75th percentile of
                      # per-user counts (users with at least one action in that channel, train)

MIN_ITEM_COUNT = 2     # items seen once in train histories get the shared "unknown" embedding
ATTR_COLS = ["brand_id", "cate_id_4", "shop_id"]

TERM_MIN_COUNT = 2     # query/name terms seen once in train map to "unknown"
QRY_MAX_TERMS = 8      # 99th percentile of query length (interim report Part 2 §2.6)
NAME_MAX_TERMS = 50    # 99th percentile of product name length (p95 = 42, p99 = 50)

ACTIONS = ["CLICK", "FLW", "CART", "ORD"]                       # action ids 1..4 (0 = padding)
TIME_BINS_DAYS = [1 / 24, 1, 3, 7, 14, 30, 60, 90, 180]         # time buckets 1..10 (0 = padding)

PAD, UNK = 0, 1
NO_QUERY = "-1"
TERM_SEP = "\x18"

HIST_COLS = ["history_wid_list", "history_type_list", "history_has_query",
             "history_days_before", "history_qry_list"]
CAND_COLS = ["candidate_wid_list", "candidate_label_list", "candidate_has_metadata"]

## Helpers

In [ ]:
def read_split(path):
    return pq.read_table(path, columns=["slate_id"] + HIST_COLS + CAND_COLS)


def flat(table, col):
    """List column -> (flat values as a pyarrow array, list lengths as numpy)."""
    arr = table.column(col).combine_chunks()
    return arr.flatten(), arr.value_lengths().to_numpy(zero_copy_only=False)


def list_lengths(arr):
    return pc.fill_null(pc.list_value_length(arr), 0).to_numpy(zero_copy_only=False)


class Lookup:
    """Maps raw IDs to embedding indices with a sorted binary search; misses -> UNK."""

    def __init__(self, keys, values=None):
        keys = np.asarray(keys)
        values = np.arange(2, len(keys) + 2) if values is None else np.asarray(values)
        order = np.argsort(keys, kind="stable")
        self.keys, self.values = keys[order], values[order]

    def __call__(self, ids):
        ids = np.asarray(ids)
        if len(self.keys) == 0:
            return np.full(ids.shape, UNK)
        pos = np.minimum(np.searchsorted(self.keys, ids), len(self.keys) - 1)
        return np.where(self.keys[pos] == ids, self.values[pos], UNK)


def split_terms(strings):
    """Arrow array of TERM_SEP-joined strings -> (flat int64 term ids, terms per string).
    Empty terms (from a trailing or doubled separator, or an empty string) are dropped."""
    lists = pc.split_pattern(strings, TERM_SEP)
    terms = lists.flatten()
    keep = pc.not_equal(terms, "").to_numpy(zero_copy_only=False)
    row = np.repeat(np.arange(len(lists)), list_lengths(lists))
    lens = np.bincount(row[keep], minlength=len(lists))
    return pc.cast(terms.filter(pa.array(keep)), pa.int64()).to_numpy(), lens


def gather_ranges(starts, lens):
    """Flat positions for rows given their start offsets and lengths."""
    within = np.arange(lens.sum()) - np.repeat(np.cumsum(lens) - lens, lens)
    return np.repeat(starts, lens) + within


def pad_terms(term_idx, lens, max_terms):
    """Flat term indices + row lengths -> [n_rows, max_terms], keeping the first terms."""
    n = len(lens)
    row = np.repeat(np.arange(n), lens)
    col = np.arange(len(term_idx)) - np.repeat(np.cumsum(lens) - lens, lens)
    keep = col < max_terms
    out = np.zeros((n, max_terms), dtype=np.int32)
    out[row[keep], col[keep]] = term_idx[keep]
    return out


def decayed_recency(days, half_life_days):
    """Exponential decay with the given half-life. Applied at training time to a channel's
    `days` array with that channel's half-life from vocab.json (padding is removed by the mask),
    so half-lives can be tuned on validation without rebuilding the inputs."""
    return np.exp(-np.log(2) * days / half_life_days).astype(np.float32)

## Step 1a: Load product names for every candidate

The product file is large (mostly names), so it is streamed in batches and only the
products appearing as candidates in train/val/test are kept, already truncated to
NAME_MAX_TERMS. This is catalogue lookup data, not learned, so including val/test
products here is not leakage; the term vocabulary below still comes from train only.

In [ ]:
class ProductNames:
    def __init__(self, path, needed_wids, batch_size=500_000):
        needed = pa.array(np.unique(needed_wids))
        wids, terms, lens, full_lens = [], [], [], []
        for b in pq.ParquetFile(path).iter_batches(columns=["wid", "name"], batch_size=batch_size):
            b = b.filter(pc.is_in(b.column("wid"), value_set=needed.cast(b.schema.field("wid").type)))
            if b.num_rows == 0:
                continue
            name = b.column("name")
            raw = pc.cast(name.flatten(), pa.int64()).to_numpy()
            l = list_lengths(name)
            keep = (np.arange(len(raw)) - np.repeat(np.cumsum(l) - l, l)) < NAME_MAX_TERMS
            wids.append(b.column("wid").to_numpy())
            terms.append(raw[keep])
            lens.append(np.minimum(l, NAME_MAX_TERMS))
            full_lens.append(l)
        wid = np.concatenate(wids)
        lens = np.concatenate(lens)
        offsets = np.cumsum(lens) - lens
        order = np.argsort(wid, kind="stable")
        self.wid, self.lens, self.offsets = wid[order], lens[order], offsets[order]
        self.terms = np.concatenate(terms)
        full = np.concatenate(full_lens)
        print(f"product names loaded: {len(wid):,}  name length p50/p90/p95/p99 = "
              f"{np.percentile(full, [50, 90, 95, 99]).round(0).tolist()}  "
              f"(truncated at {NAME_MAX_TERMS}: {(full > NAME_MAX_TERMS).mean():.1%})")

    def raw_terms(self, wids):
        """wids -> (flat raw term ids, length per wid); unknown wids get length 0."""
        wids = np.asarray(wids)
        pos = np.minimum(np.searchsorted(self.wid, wids), len(self.wid) - 1)
        hit = self.wid[pos] == wids
        lens = np.where(hit, self.lens[pos], 0)
        return self.terms[gather_ranges(self.offsets[pos], lens)], lens

## Step 1b: Build vocabularies from the train split

Item vocabulary: items appearing at least MIN_ITEM_COUNT times in train **histories**.
Candidates are deliberately not counted: a train candidate counting itself would make
candidates look "known" more often in train than in val/test. Attribute vocabularies: brand / category / shop values of the
products seen in train; the code -1 (missing) is left out so it maps to unknown. Term vocabulary: terms appearing at least TERM_MIN_COUNT times
across train history queries and train candidate names (one shared vocabulary, so a
query term and a name term with the same ID share an embedding).

In [ ]:
def build_vocabs(train_table, product, names):
    hist_wids, _ = flat(train_table, "history_wid_list")
    cand_wids, _ = flat(train_table, "candidate_wid_list")
    # Items: counted in train HISTORIES only, so "known" means the same thing for train,
    # val and test candidates (a train candidate must not count itself into the vocabulary).
    hist_days = flat(train_table, "history_days_before")[0].to_numpy()
    h_uniq, h_counts = np.unique(hist_wids.to_numpy()[hist_days >= MIN_AGE_DAYS], return_counts=True)
    item_vocab = h_uniq[h_counts >= MIN_ITEM_COUNT]

    prod_wid = product.column("wid").to_numpy()
    seen_in_train = np.isin(prod_wid, np.union1d(h_uniq, np.unique(cand_wids.to_numpy())))
    attr_vocabs = {}
    for col in ATTR_COLS:
        vals = np.unique(product.column(col).to_numpy()[seen_in_train])
        attr_vocabs[col] = vals[vals >= 0]            # -1 = missing in the product file -> UNK

    qry = flat(train_table, "history_qry_list")[0]
    qry_terms, _ = split_terms(qry.filter(pc.not_equal(qry, NO_QUERY)))
    name_terms, _ = names.raw_terms(cand_wids.to_numpy())
    t_uniq, t_counts = np.unique(np.concatenate([qry_terms, name_terms]), return_counts=True)
    term_vocab = t_uniq[t_counts >= TERM_MIN_COUNT]

    q_set, n_set = np.unique(qry_terms), np.unique(name_terms)
    print(f"term vocab: {len(term_vocab):,} terms  |  distinct query terms {len(q_set):,}, "
          f"name terms {len(n_set):,}; share of query terms also used in names: "
          f"{np.isin(q_set, n_set).mean():.1%}")
    return item_vocab, attr_vocabs, term_vocab


class ProductAttrs:
    """wid -> attribute indices. Products without metadata, or with an attribute value
    not in the train vocabulary, get UNK."""

    def __init__(self, product, attr_vocabs):
        wid = product.column("wid").to_numpy()
        order = np.argsort(wid, kind="stable")
        self.wid = wid[order]
        self.attr_idx = {col: Lookup(attr_vocabs[col])(product.column(col).to_numpy()[order])
                         for col in ATTR_COLS}

    def __call__(self, wids):
        wids = np.asarray(wids)
        pos = np.minimum(np.searchsorted(self.wid, wids), len(self.wid) - 1)
        hit = self.wid[pos] == wids
        return {col: np.where(hit, idx[pos], UNK).astype(np.int32) for col, idx in self.attr_idx.items()}

## Step 1c: Per-channel caps and recency scales (from train)

For each channel, on the train split:

- **cap** = the 75th percentile of per-user action counts among users with at least one action in that channel. The longest 25% of each channel are truncated to their most recent actions, the same rule the interim report used for the whole history, now applied per window.
- **half-life** = the median age of that channel's actions, so a typical action gets a recency weight of 0.5. A single half-life for all actions would mean nothing: clicks are at most 7 days old and orders up to a year.

Actions older than their channel's window (the few stragglers in the table above) are dropped, so each channel covers exactly its window. Actions just before the search are kept (`MIN_AGE_DAYS = 0`, the same rule as Model 1); setting it above 0 drops them for a sensitivity check.


In [ ]:
def channel_masks(table):
    """Flat per-action days and, per channel, boolean masks (kept, older than window, too recent).
    Kept = action type in the channel, at least MIN_AGE_DAYS before the search, within the window."""
    types = flat(table, "history_type_list")[0]
    days = flat(table, "history_days_before")[0].to_numpy()
    masks = {}
    for ch, (acts, window) in CHANNELS.items():
        of_type = pc.is_in(types, value_set=pa.array(acts)).to_numpy(zero_copy_only=False)
        too_recent = of_type & (days < MIN_AGE_DAYS)
        masks[ch] = (of_type & ~too_recent & (days <= window), of_type & (days > window), too_recent)
    return days, masks


def channel_settings(train_table):
    _, lens = flat(train_table, "history_wid_list")
    row = np.repeat(np.arange(train_table.num_rows), lens)
    days, masks = channel_masks(train_table)
    settings = {}
    for ch, (acts, window) in CHANNELS.items():
        inside, outside, too_recent = masks[ch]
        counts = np.bincount(row[inside], minlength=train_table.num_rows)
        settings[ch] = {
            "actions": acts,
            "window_days": window,
            "cap": int(np.ceil(np.percentile(counts[counts > 0], CAP_PERCENTILE))),
            "half_life_days": round(float(np.median(days[inside])), 2),
            "share_users_with_any": round(float((counts > 0).mean()), 4),
            "dropped_outside_window": int(outside.sum()),
            "dropped_near_query": int(too_recent.sum()),
        }
    return settings

## Step 2: Encode one split

Each channel is encoded separately: select that channel's actions (inside its window), give each its distance from the end of the user's channel (most recent = 0), keep distances below the channel's cap, and scatter them into column `cap - 1 - distance`. This truncates and left-pads in one vectorised step, so in every channel the last column is the most recent action.

Per channel and position: item index, brand / L4-category / shop index, action type (needed in the cart channel to tell CART from FLW), searched-or-not flag, the query that led to the action, days before the search, and a log-scale time bucket. Per channel and slate: number of actions in the window (`full_len`), number kept (`len`) and days since the most recent one (`days_since_last`, missing if none).

Past queries are stored once per distinct query string (`qry_terms`, row 0 = no query) and referenced by row number; candidate names likewise (`name_terms`). Step 3b expands both into the saved file.

In [ ]:
def encode_split(table, item_lookup, prod_attrs, term_lookup, names, settings):
    n = table.num_rows

    # ---- flat per-action arrays, shared by all channels ----
    wids, lens = flat(table, "history_wid_list")
    wids = wids.to_numpy()
    row = np.repeat(np.arange(n), lens)
    action_ids = pc.index_in(flat(table, "history_type_list")[0], value_set=pa.array(ACTIONS))
    assert action_ids.null_count == 0, "unexpected value in history_type_list"
    action_ids = action_ids.to_numpy() + 1
    has_query = flat(table, "history_has_query")[0].to_numpy(zero_copy_only=False)
    days, masks = channel_masks(table)

    # past queries -> rows of a per-split query term table
    qry = pc.dictionary_encode(flat(table, "history_qry_list")[0])
    qry_dict = qry.dictionary
    is_noq = pc.equal(qry_dict, NO_QUERY).to_numpy(zero_copy_only=False)
    q_raw, q_lens = split_terms(qry_dict)
    q_table = pad_terms(term_lookup(q_raw).astype(np.int32), q_lens, QRY_MAX_TERMS)
    q_table[is_noq] = PAD
    qry_terms = np.vstack([np.zeros((1, QRY_MAX_TERMS), np.int32), q_table])
    q_idx = qry.indices.to_numpy() + 1
    q_idx[is_noq[q_idx - 1]] = 0

    per_action = {
        "item": (item_lookup(wids), np.int32),
        "action": (action_ids, np.int8),
        "has_query": (has_query, np.int8),
        "qry": (q_idx, np.int32),
        "days": (days, np.float32),
        "time_bucket": (np.digitize(days, TIME_BINS_DAYS) + 1, np.int8),
        **{col: (idx, np.int32) for col, idx in prod_attrs(wids).items()},
    }

    # ---- one set of grids per channel ----
    seqs = {}
    for ch in CHANNELS:
        cap = settings[ch]["cap"]
        sel = np.flatnonzero(masks[ch][0])               # this channel's actions, in order
        r_sel = row[sel]
        counts = np.bincount(r_sel, minlength=n)
        pos_from_end = np.repeat(np.cumsum(counts), counts) - 1 - np.arange(len(sel))
        keep = pos_from_end < cap
        r, c, src = r_sel[keep], cap - 1 - pos_from_end[keep], sel[keep]

        s = {}
        for field, (values, dtype) in per_action.items():
            grid = np.zeros((n, cap), dtype=dtype)
            grid[r, c] = values[src]
            s[field] = grid
        last = pos_from_end == 0
        s["days_since_last"] = np.full(n, np.nan, dtype=np.float32)
        s["days_since_last"][r_sel[last]] = days[sel[last]]
        s["len"] = np.minimum(counts, cap).astype(np.int16)
        s["full_len"] = counts.astype(np.int32)
        seqs[ch] = s

    user = {"history_len": lens.astype(np.int32)}      # all actions, before any windowing

    # ---- candidates (flat, one entry per candidate) ----
    cwids, clens = flat(table, "candidate_wid_list")
    cwids = cwids.to_numpy()
    starts = np.repeat(np.cumsum(clens) - clens, clens)
    u_wids, name_row = np.unique(cwids, return_inverse=True)
    n_raw, n_lens = names.raw_terms(u_wids)
    name_terms = pad_terms(term_lookup(n_raw).astype(np.int32), n_lens, NAME_MAX_TERMS)

    cand = {
        "row": np.repeat(np.arange(n), clens).astype(np.int32),
        "pos": (np.arange(len(cwids)) - starts).astype(np.int16),
        "item": item_lookup(cwids).astype(np.int32),
        "name": name_row.astype(np.int32),
        "label": flat(table, "candidate_label_list")[0].to_numpy().astype(np.int8),
        "has_metadata": flat(table, "candidate_has_metadata")[0].to_numpy(zero_copy_only=False).astype(np.int8),
    }
    cand.update(prod_attrs(cwids))

    tables = {"qry_terms": qry_terms, "name_terms": name_terms}
    slate_id = table.column("slate_id").to_numpy()
    return slate_id, user, seqs, cand, tables

## Step 3: Sanity checks
Stops with an error if any channel is malformed, and prints per-channel coverage plus the unknown-item and unknown-term rates, which are worth reporting.

In [ ]:
def check_split(name, table, slate_id, user, seqs, cand, tables, settings):
    print(f"[{name}] slates={len(slate_id):,}  candidates={len(cand['item']):,}")
    action_sets = {ch: [ACTIONS.index(a) + 1 for a in acts] for ch, (acts, _) in CHANNELS.items()}
    for ch, s in seqs.items():
        cap, window = settings[ch]["cap"], settings[ch]["window_days"]
        mask = s["action"] > 0
        assert (mask.sum(1) == np.minimum(s["full_len"], cap)).all(), f"{ch}: mask length mismatch"
        assert (s["item"][~mask] == PAD).all(), f"{ch}: non-zero item in padding"
        assert (mask[:, -1] == (s["full_len"] > 0)).all(), f"{ch}: most recent action not in the last column"
        assert np.isin(s["action"][mask], action_sets[ch]).all(), f"{ch}: wrong action type in channel"
        assert (s["days"][mask] <= window).all(), f"{ch}: action outside its window"
        assert (s["days"][mask] >= MIN_AGE_DAYS).all(), f"{ch}: near-query action kept"
        assert ((s["qry"] > 0) == (s["has_query"] == 1)).all(), f"{ch}: query rows disagree with has_query"
        assert np.array_equal(np.isnan(s["days_since_last"]), s["full_len"] == 0), f"{ch}: days_since_last mismatch"
        has_any = s["full_len"] > 0
        print(f"[{name}] {ch:5s} (≤{window}d, cap {cap}): users with any {has_any.mean():.1%} | "
              f"truncated {(s['full_len'] > cap).mean():.1%} of users | "
              f"unknown items {(s['item'][mask] == UNK).mean():.1%}")
    windowed = sum(s["full_len"] for s in seqs.values())
    print(f"[{name}] actions dropped (outside window or < {MIN_AGE_DAYS} days before the search): "
          f"{(user['history_len'] - windowed).sum():,}")

    src_labels = flat(table, "candidate_label_list")[0].to_numpy()
    assert (cand["label"] == src_labels).all(), "candidate labels changed"
    cand_unk = (cand["item"] == UNK).mean()
    buy_unk = (cand["item"][cand["label"] == 3] == UNK).mean() if (cand["label"] == 3).any() else float("nan")
    print(f"[{name}] unknown candidate items: all {cand_unk:.1%} | purchased {buy_unk:.1%}")
    for col in ATTR_COLS:
        print(f"[{name}] unknown {col}: candidates {(cand[col] == UNK).mean():.1%}")
    qt, nt = tables["qry_terms"], tables["name_terms"]
    no_name = (nt[cand["name"]] == PAD).all(1).mean()
    print(f"[{name}] unknown terms: queries {(qt[qt > 0] == UNK).mean():.1%} | "
          f"names {(nt[nt > 0] == UNK).mean():.1%}  |  candidates with no name: {no_name:.1%}")

## Step 3b: Saving and loading as parquet

Each split is saved as **one parquet file** (`seq_train.parquet`, `seq_val1.parquet`, `seq_val2.parquet`, `seq_test.parquet`) with **one row per slate**, the same layout as the step 1 files:

| Column group | Per row (one slate) |
|---|---|
| `slate_id`, `history_len` | single values |
| `<channel>_len`, `<channel>_full_len`, `<channel>_days_since_last` | single values per channel (`click`, `cart`, `order`) |
| `<channel>_item`, `<channel>_action`, `<channel>_days`, … | a list of `cap` values (that channel's grid row) |
| `<channel>_qry_terms` | `cap` lists of `QRY_MAX_TERMS` term indices (the query behind each action) |
| `cand_item`, `cand_label`, `cand_brand_id`, … | a list with one value per candidate, in step 1 order |
| `cand_name_terms` | one list of `NAME_MAX_TERMS` term indices per candidate |

Histories are stored once per slate, never copied per candidate. `load_split` reads a file back into model-ready arrays: per-channel grids `[n_slates, cap]`, and flat candidate arrays with `row` (the candidate's slate row) and `pos` (its position in the slate). `(slate_id, pos)` is the join key with the other feature files.

In [ ]:
def fixed_list(grid):
    """numpy array [n, k] or [n, k, m] -> Arrow fixed-size list column (nested for 3-D)."""
    values = pa.array(np.ascontiguousarray(grid).reshape(-1))
    for size in reversed(grid.shape[1:]):
        values = pa.FixedSizeListArray.from_arrays(values, size)
    return values


def to_grid(arr):
    """Arrow (nested) fixed-size list array -> numpy array [n, k] or [n, k, m]."""
    shape, t = [len(arr)], arr.type
    while pa.types.is_fixed_size_list(t):
        shape.append(t.list_size)
        arr, t = arr.flatten(), t.value_type
    return arr.to_numpy(zero_copy_only=False).reshape(shape)


def model_arrays(slate_id, user, seqs, cand, tables):
    """Resolve the query/name pointers into the term indices themselves."""
    seqs = {ch: {k: v for k, v in s.items() if k != "qry"} | {"qry_terms": tables["qry_terms"][s["qry"]]}
            for ch, s in seqs.items()}
    cand = {k: v for k, v in cand.items() if k != "name"} | {"name_terms": tables["name_terms"][cand["name"]]}
    return slate_id, user, seqs, cand


def save_split(name, slate_id, user, seqs, cand, out_dir=None):
    """One parquet file per split, one row per slate (takes the output of model_arrays)."""
    out_dir = out_dir or OUT_DIR
    cols = {"slate_id": slate_id, **user}
    for ch, s in seqs.items():
        for k, v in s.items():
            cols[f"{ch}_{k}"] = v if v.ndim == 1 else fixed_list(v)
    counts = np.bincount(cand["row"], minlength=len(slate_id))
    offsets = pa.array(np.concatenate([[0], np.cumsum(counts)]).astype(np.int32))
    for k, v in cand.items():
        if k in ("row", "pos"):          # implied by the list layout
            continue
        cols[f"cand_{k}"] = pa.ListArray.from_arrays(offsets, pa.array(v) if v.ndim == 1 else fixed_list(v))
    pq.write_table(pa.table(cols), out_dir / f"seq_{name}.parquet", compression="zstd")


def load_split(name, out_dir=None):
    """Read one split back as (slate_id, user, seqs, cand): per-channel grids per slate, and flat
    candidate arrays with row (slate row) and pos (position within the slate)."""
    out_dir = out_dir or OUT_DIR
    t = pq.read_table(out_dir / f"seq_{name}.parquet")
    user, seqs, cand = {}, {ch: {} for ch in CHANNELS}, {}
    for col in t.column_names:
        arr = t.column(col).combine_chunks()
        prefix, _, field = col.partition("_")
        if prefix in CHANNELS:
            seqs[prefix][field] = to_grid(arr) if pa.types.is_fixed_size_list(arr.type) \
                                  else arr.to_numpy(zero_copy_only=False)
        elif prefix == "cand":
            values = arr.flatten()
            cand[field] = to_grid(values) if pa.types.is_fixed_size_list(values.type) else values.to_numpy()
            if "row" not in cand:
                counts = arr.value_lengths().to_numpy()
                cand["row"] = np.repeat(np.arange(len(arr)), counts).astype(np.int32)
                cand["pos"] = (np.arange(counts.sum()) - np.repeat(np.cumsum(counts) - counts, counts)).astype(np.int16)
        elif col != "slate_id":
            user[col] = arr.to_numpy()
    return t.column("slate_id").to_numpy(), user, seqs, cand


def assert_same(name, expected, loaded, path=""):
    """Round-trip check: what was saved reads back identically (NaN counts as equal)."""
    if isinstance(expected, (tuple, list)):
        for i, (e, l) in enumerate(zip(expected, loaded)):
            assert_same(name, e, l, f"{path}[{i}]")
    elif isinstance(expected, dict):
        assert set(expected) == set(loaded), f"[{name}] {path}: columns differ {set(expected) ^ set(loaded)}"
        for k in expected:
            assert_same(name, expected[k], loaded[k], f"{path}.{k}")
    else:
        assert np.array_equal(expected, loaded, equal_nan=expected.dtype.kind == "f"), \
            f"[{name}] {path} changed when saved"

## Step 4: Run the pipeline

Train features use `train_sampled.parquet` (negative-sampled lists); val1, val2 and test use the full lists, so model selection, calibration and evaluation are not made artificially easy.

### 4.1 Load the step 1 outputs

In [ ]:
# Remove this notebook's previous outputs (only its own files: the folder also holds
# other files), so it only ever holds one complete run
OUT_DIR.mkdir(parents=True, exist_ok=True)
OWN_OUTPUTS = ["seq_*.parquet", "vocab_*.parquet", "vocab.json"]
for pattern in OWN_OUTPUTS:
    for f in OUT_DIR.glob(pattern):
        f.unlink()
product_path = DATA_DIR / "product_metadata_clean.parquet"
product = pq.read_table(product_path, columns=["wid"] + ATTR_COLS)

splits = {name: DATA_DIR / f for name, f in SPLIT_FILES.items()}
tables = {name: read_split(path) for name, path in splits.items()}
for name, t in tables.items():
    print(f"{name}: {t.num_rows:,} slates")

train: 122,083 slates
val1: 17,799 slates
val2: 16,924 slates
test: 17,025 slates


### 4.2 Check the logging windows on the train split

Reproduces the finding behind the three channels: the share of each action type by age band, and how far back each type reaches. Clicks should stop at about 7 days, carts and follows at about 90, orders at about 365.

In [ ]:
types_tr = flat(tables["train"], "history_type_list")[0].to_numpy(zero_copy_only=False)
days_tr = flat(tables["train"], "history_days_before")[0].to_numpy()
bands = pd.cut(days_tr, [0, 1, 3, 7, 14, 30, 60, 90, 180, 270, 365, np.inf], right=False)
mix = pd.crosstab(bands, types_tr).reindex(columns=["CLICK", "CART", "FLW", "ORD"])
print("Share of each action type by age band (days before the search):")
print(mix.div(mix.sum(axis=1), axis=0).round(3))
print("\nHow far back each type reaches (days):")
print(pd.Series(days_tr).groupby(types_tr).describe(percentiles=[.5, .9, .99]).round(1))
del types_tr, days_tr

Share of each action type by age band (days before the search):
col_0           CLICK   CART    FLW    ORD
row_0                                     
[0.0, 1.0)      0.856  0.128  0.000  0.016
[1.0, 3.0)      0.854  0.126  0.000  0.019
[3.0, 7.0)      0.831  0.145  0.000  0.024
[7.0, 14.0)     0.000  0.858  0.002  0.140
[14.0, 30.0)    0.000  0.854  0.002  0.145
[30.0, 60.0)    0.000  0.848  0.002  0.150
[60.0, 90.0)    0.000  0.834  0.002  0.164
[90.0, 180.0)   0.000  0.001  0.000  0.999
[180.0, 270.0)  0.000  0.000  0.000  1.000
[270.0, 365.0)  0.000  0.000  0.000  1.000

How far back each type reaches (days):
           count   mean    std  min    50%    90%    99%    max
CART   8084963.0   40.0   25.9  0.0   38.3   78.2   88.3  160.1
CLICK  5840341.0    2.9    2.0  0.0    2.8    5.7    6.5    7.0
FLW      16284.0   40.6   26.3  0.0   39.8   79.9   87.7   89.9
ORD    4778769.0  165.8  106.0  0.0  154.1  320.0  359.6  365.0


### 4.3 Load product names for all candidates (streamed in batches)

In [ ]:
all_cands = np.concatenate([flat(t, "candidate_wid_list")[0].to_numpy() for t in tables.values()])
names = ProductNames(product_path, all_cands)
del all_cands

product names loaded: 4,541,808  name length p50/p90/p95/p99 = [28.0, 39.0, 42.0, 50.0]  (truncated at 50: 0.9%)


### 4.4 Build vocabularies and channel settings from train, and save them

`vocab.json` records every vocabulary size (for the embedding tables), each channel's window, cap and recency half-life, and the other settings, so the model code and the report refer to one source.

In [ ]:
item_vocab, attr_vocabs, term_vocab = build_vocabs(tables["train"], product, names)
item_lookup, term_lookup = Lookup(item_vocab), Lookup(term_vocab)
prod_attrs = ProductAttrs(product, attr_vocabs)
settings = channel_settings(tables["train"])

# Each vocabulary file lists the raw IDs in index order: raw_id in row i has embedding index i + 2
for vname, v in {"item": item_vocab, "term": term_vocab, **attr_vocabs}.items():
    vname = vname.replace("_id", "")
    pq.write_table(pa.table({"raw_id": np.sort(v)}), OUT_DIR / f"vocab_{vname}.parquet")

vocab_info = {
    "n_items": int(len(item_vocab) + 2),
    **{f"n_{col}": int(len(v) + 2) for col, v in attr_vocabs.items()},
    "n_terms": int(len(term_vocab) + 2),
    "n_actions": len(ACTIONS) + 1,
    "n_time_buckets": len(TIME_BINS_DAYS) + 2,
    "channels": settings,
    "cap_percentile": CAP_PERCENTILE, "min_item_count": MIN_ITEM_COUNT,
    "term_min_count": TERM_MIN_COUNT,
    "qry_max_terms": QRY_MAX_TERMS, "name_max_terms": NAME_MAX_TERMS,
    "actions": ACTIONS, "time_bins_days": TIME_BINS_DAYS,
}
(OUT_DIR / "vocab.json").write_text(json.dumps(vocab_info, indent=2))
print(json.dumps(vocab_info, indent=2))

term vocab: 320,665 terms  |  distinct query terms 240,265, name terms 420,360; share of query terms also used in names: 52.8%
{
  "n_items": 1873780,
  "n_brand_id": 152583,
  "n_cate_id_4": 6760,
  "n_shop_id": 226462,
  "n_terms": 320667,
  "n_actions": 5,
  "n_time_buckets": 11,
  "channels": {
    "click": {
      "actions": [
        "CLICK"
      ],
      "window_days": 7,
      "cap": 65,
      "half_life_days": 2.78,
      "share_users_with_any": 0.8369,
      "dropped_outside_window": 0,
      "dropped_near_query": 0
    },
    "cart": {
      "actions": [
        "CART",
        "FLW"
      ],
      "window_days": 90,
      "cap": 80,
      "half_life_days": 38.34,
      "share_users_with_any": 0.9562,
      "dropped_outside_window": 841,
      "dropped_near_query": 0
    },
    "order": {
      "actions": [
        "ORD"
      ],
      "window_days": 365,
      "cap": 49,
      "half_life_days": 154.11,
      "share_users_with_any": 0.9159,
      "dropped_outside_window": 0

### 4.5 Encode, check and save each split

Each split is saved as one parquet file (`seq_<split>.parquet`, Step 3b) and read back to confirm nothing changed. The printed per-channel coverage and unknown rates are worth reporting in Section 1.

In [ ]:
for name, table in tables.items():
    slate_id, user, seqs, cand, term_tables = encode_split(table, item_lookup, prod_attrs, term_lookup, names, settings)
    check_split(name, table, slate_id, user, seqs, cand, term_tables, settings)
    arrays = model_arrays(slate_id, user, seqs, cand, term_tables)
    save_split(name, *arrays)
    assert_same(name, arrays, load_split(name))
    print(f"[{name}] saved to seq_{name}.parquet and read back identically\n")
    del slate_id, user, seqs, cand, term_tables, arrays

[train] slates=122,083  candidates=2,608,369
[train] click (≤7d, cap 65): users with any 83.7% | truncated 20.9% of users | unknown items 8.5%
[train] cart  (≤90d, cap 80): users with any 95.6% | truncated 23.8% of users | unknown items 15.3%
[train] order (≤365d, cap 49): users with any 91.6% | truncated 22.7% of users | unknown items 11.5%
[train] actions dropped (outside window or < 0.0 days before the search): 841
[train] unknown candidate items: all 65.7% | purchased 19.3%
[train] unknown brand_id: candidates 0.5%
[train] unknown cate_id_4: candidates 0.5%
[train] unknown shop_id: candidates 1.5%
[train] unknown terms: queries 1.6% | names 0.2%  |  candidates with no name: 0.5%
[train] saved to seq_train.parquet and read back identically

[val1] slates=17,799  candidates=1,589,673
[val1] click (≤7d, cap 65): users with any 83.3% | truncated 20.3% of users | unknown items 32.0%
[val1] cart  (≤90d, cap 80): users with any 95.8% | truncated 23.3% of users | unknown items 30.1%
[val1]

## Step 5: Inspect the saved output

Reads the validation split back with `load_split` and prints every array's shape and type (what the model will receive), plus the size of each output file.

In [ ]:
slate_id_v, user_v, seqs_v, cand_v = load_split("val2")
shown = {"slate_id": slate_id_v, **user_v,
         **{f"{ch}_{k}": v for ch, s in seqs_v.items() for k, v in s.items()},
         **{f"cand_{k}": v for k, v in cand_v.items()}}
for k, v in shown.items():
    print(f"{k:26s} {str(v.shape):22s} {v.dtype}")

print("\nfiles:")
for f in sorted(f for pattern in OWN_OUTPUTS for f in OUT_DIR.glob(pattern)):
    print(f"  {f.name:28s} {f.stat().st_size / 1e6:8.1f} MB")

slate_id                   (16924,)               int32
history_len                (16924,)               int32
click_item                 (16924, 65)            int32
click_action               (16924, 65)            int8
click_has_query            (16924, 65)            int8
click_days                 (16924, 65)            float32
click_time_bucket          (16924, 65)            int8
click_brand_id             (16924, 65)            int32
click_cate_id_4            (16924, 65)            int32
click_shop_id              (16924, 65)            int32
click_days_since_last      (16924,)               float32
click_len                  (16924,)               int16
click_full_len             (16924,)               int32
click_qry_terms            (16924, 65, 8)         int32
cart_item                  (16924, 80)            int32
cart_action                (16924, 80)            int8
cart_has_query             (16924, 80)            int8
cart_days                  (16924, 80)           